# 04 - Knowledge Base
Create an Aurora pgvector pgvector table, Bedrock Knowledge Base with S3 data source, and test RAG retrieval.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
from pathlib import Path

config_path = Path("config.json")
config = json.loads(config_path.read_text()) if config_path.exists() else {}

REGION     = config.get("REGION", "us-east-1")
PREFIX     = config.get("PREFIX", "orgiq")
ACCOUNT_ID = config.get("ACCOUNT_ID", boto3.client("sts").get_caller_identity()["Account"])

# ── CDK stack outputs (populated by notebook 01 Step 6) ──────────────
AURORA_CLUSTER_ARN = config.get("AURORA_CLUSTER_ARN", "")
AURORA_SECRET_ARN  = config.get("AURORA_SECRET_ARN",  "")
DB_NAME            = config.get("AURORA_DB_NAME",   "orgiq")
DOCUMENTS_BUCKET   = config.get("DOCUMENTS_BUCKET",  "")
KB_ID              = config.get("KB_ID", "")

missing = [k for k, v in {
    "AURORA_CLUSTER_ARN": AURORA_CLUSTER_ARN,
    "AURORA_SECRET_ARN":  AURORA_SECRET_ARN,
    "DOCUMENTS_BUCKET":   DOCUMENTS_BUCKET,
    "KB_ID":              KB_ID,
}.items() if not v]

if missing:
    raise SystemExit(
        f"✗ Missing config keys: {missing}\n"
        "  Run notebook 01 Step 6 first to load CDK outputs.\n"
        "  Requires OrgIQ-Foundation, OrgIQ-Aurora, OrgIQ-KnowledgeBase to be deployed."
    )

rds = boto3.client("rds-data", region_name=REGION)
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)

print(f"✓ Config loaded from CDK outputs")
print(f"  AURORA_CLUSTER_ARN : {AURORA_CLUSTER_ARN.split(':cluster:')[-1]}")
print(f"  DOCUMENTS_BUCKET   : {DOCUMENTS_BUCKET}")
print(f"  KB_ID              : {KB_ID}")

## Step 2: Create S3 Bucket for Documents

In [ ]:
# Bucket name comes from OrgIQ-Foundation CDK output
BUCKET_NAME = DOCUMENTS_BUCKET
s3 = boto3.client("s3", region_name=REGION)
print(f"✓ Using CDK-deployed documents bucket: {BUCKET_NAME}")

## Step 3: Upload Sample Documents

In [ ]:
sample_docs = [
    {
        "key": "docs/kubernetes-migration.md",
        "content": """# Kubernetes Migration Playbook\n\nThis guide covers migrating monolithic applications to Kubernetes.\n\n## Prerequisites\n- Container runtime (Docker/containerd)\n- kubectl configured\n- Cluster with 3+ nodes\n\n## Steps\n1. Containerize the application with multi-stage Dockerfiles\n2. Define resource requests and limits\n3. Create Deployment and Service manifests\n4. Configure Horizontal Pod Autoscaler\n5. Set up Ingress with TLS termination\n\n## Best Practices\n- Use namespaces for isolation\n- Implement readiness and liveness probes\n- Store secrets in AWS Secrets Manager\n- Use Karpenter for node autoscaling"""
    },
    {
        "key": "docs/ml-pipeline-architecture.md",
        "content": """# ML Pipeline Architecture\n\nEnd-to-end machine learning pipeline using SageMaker and Step Functions.\n\n## Components\n- Data ingestion: S3 + Glue ETL\n- Feature store: SageMaker Feature Store\n- Training: SageMaker Training Jobs\n- Evaluation: Custom metrics + SageMaker Experiments\n- Deployment: Multi-model endpoints\n\n## Model Registry\nAll models versioned in SageMaker Model Registry with approval workflows.\n\n## Monitoring\n- Model drift detection with SageMaker Model Monitor\n- A/B testing via traffic splitting\n- CloudWatch custom metrics for business KPIs"""
    },
    {
        "key": "docs/zero-trust-auth.md",
        "content": """# Zero Trust Authentication Design\n\nImplementing zero trust security for microservices.\n\n## Principles\n- Never trust, always verify\n- Least privilege access\n- Assume breach\n\n## Implementation\n1. mTLS between all services (Istio service mesh)\n2. JWT validation at every service boundary\n3. Short-lived tokens (15 min expiry)\n4. Device trust scoring\n5. Continuous authorization checks\n\n## Token Architecture\n- Identity Provider: AWS Cognito + OIDC federation\n- Access tokens: RS256 signed JWTs\n- Refresh tokens: stored in DynamoDB with rotation\n- Service tokens: AWS IAM roles for service accounts"""
    },
]

for doc in sample_docs:
    s3.put_object(Bucket=BUCKET_NAME, Key=doc["key"], Body=doc["content"].encode())
    print(f"  \u2713 Uploaded: {doc['key']}")

print(f"\u2713 {len(sample_docs)} documents uploaded to S3")

## Step 4: Initialize Aurora pgvector Table

Creates the `bedrock_kb` table with a 1024-dimension HNSW vector index.
Run once after the Aurora cluster is up (from notebook 02 or CDK deploy).

In [ ]:
# Initialize pgvector table in Aurora via RDS Data API
# Bedrock KB expects: id (uuid PK), embedding (vector(1024)), text (text), metadata (json)

rds_data = boto3.client("rds-data", region_name=REGION)

AURORA_CLUSTER_ARN = config["AURORA_CLUSTER_ARN"]
AURORA_SECRET_ARN  = config["AURORA_SECRET_ARN"]
DB_NAME            = config.get("AURORA_DB_NAME", "orgiq")

setup_sql = [
    "CREATE EXTENSION IF NOT EXISTS vector",
    """CREATE TABLE IF NOT EXISTS bedrock_kb (
        id       uuid    PRIMARY KEY DEFAULT gen_random_uuid(),
        embedding vector(1024),
        text     text,
        metadata json
    )""",
    "CREATE INDEX IF NOT EXISTS bedrock_kb_embedding_idx ON bedrock_kb USING hnsw (embedding vector_cosine_ops)",
]

for sql in setup_sql:
    rds_data.execute_statement(
        resourceArn=AURORA_CLUSTER_ARN,
        secretArn=AURORA_SECRET_ARN,
        database=DB_NAME,
        sql=sql,
    )
    print(f"✓ {sql[:60].strip()}…" if len(sql) > 60 else f"✓ {sql.strip()}")

print("\n✓ Aurora pgvector table ready for Bedrock KB")

## Step 5: Grant Bedrock KB Role Access to Aurora Cluster

Bedrock KB service role must be allowed to call the Aurora resource policy.

In [ ]:
# Allow Bedrock KB service role on the Aurora cluster resource-based policy
# (RDS Data API uses cluster ARN; no VPC connectivity required)
rds_client = boto3.client("rds", region_name=REGION)

CLUSTER_ID = config["AURORA_CLUSTER_ARN"].split(":")[-1]

try:
    rds_client.modify_db_cluster(
        DBClusterIdentifier=CLUSTER_ID,
        EnableHttpEndpoint=True,   # Ensure Data API is enabled
        ApplyImmediately=True,
    )
    print(f"✓ Data API confirmed enabled on cluster: {CLUSTER_ID}")
except Exception as e:
    print(f"  Note: {e}")
    print("  Data API should already be enabled via CDK (enableDataApi: true)")

## Step 4: Validate CDK-Deployed Knowledge Base

The Bedrock Knowledge Base and its S3 data source are created by `OrgIQ-KnowledgeBase` CDK stack.
This cell validates they exist and are active.

In [ ]:
resp = bedrock_agent.get_knowledge_base(knowledgeBaseId=KB_ID)
kb = resp["knowledgeBase"]
status = kb["status"]

print(f"✓ Knowledge Base: {kb['name']}")
print(f"  ID     : {KB_ID}")
print(f"  Status : {status}")
print(f"  Storage: {kb['storageConfiguration']['type']}")

if status != "ACTIVE":
    print(f"\n⚠ KB status is {status}, not ACTIVE.")
    print("  Wait for OrgIQ-KnowledgeBase stack to finish deploying.")
else:
    print("\n✓ Knowledge Base is ACTIVE and ready for ingestion.")

## Step 12: Start Ingestion Job

In [ ]:
print("\u231b Starting ingestion job...")
ingestion = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=ds_id
)
job_id = ingestion["ingestionJob"]["ingestionJobId"]
print(f"  Job ID: {job_id}")

# Wait for ingestion to complete
while True:
    job_status = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=ds_id,
        ingestionJobId=job_id
    )
    status = job_status["ingestionJob"]["status"]
    if status == "COMPLETE":
        stats = job_status["ingestionJob"].get("statistics", {})
        print(f"\u2713 Ingestion complete")
        print(f"  Documents scanned: {stats.get('numberOfDocumentsScanned', 'N/A')}")
        print(f"  Documents indexed: {stats.get('numberOfNewDocumentsIndexed', 'N/A')}")
        break
    elif status == "FAILED":
        reason = job_status["ingestionJob"].get("failureReasons", ["Unknown"])
        raise RuntimeError(f"Ingestion failed: {reason}")
    else:
        print(f"  Status: {status}, waiting 15s...")
        time.sleep(15)

## Step 13: Test RAG Retrieval

In [ ]:
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)

# Test retrieval
query = "How do I migrate applications to Kubernetes?"
response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": query},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 3
        }
    }
)

print(f"Query: '{query}'")
print(f"\nRetrieved {len(response['retrievalResults'])} results:")
for i, result in enumerate(response["retrievalResults"], 1):
    score = result.get("score", 0)
    text_preview = result["content"]["text"][:150]
    print(f"\n  Result {i} (score: {score:.4f}):")
    print(f"    {text_preview}...")

print("\n\u2713 RAG retrieval working")

## Step 13b: Rerank retrieved results

The vector search above orders chunks by cosine similarity against the query
embedding. That's a decent first pass, but Titan V2 embeddings don't know
*query intent* — only semantic proximity. The top result is often the
chunk that shares the most surface-level vocabulary, not the chunk that
best answers the question.

The Bedrock **Rerank API** (`bedrock-agent-runtime.rerank`) takes the query
plus a list of candidate documents and returns a true relevance score per
document. It's a cross-encoder under the hood, so it sees the query and
the document together rather than embedding each in isolation.

Two ways to use it:

1. **Standalone** — call `rerank()` directly with any list of texts (works
   for hybrid pipelines, web results, anything you can stringify).
2. **Integrated with KB** — pass `rerankingConfiguration` to `retrieve()`
   and Bedrock does over-fetch + rerank + truncate in one call. This is
   how production agents should use it.

Model used: `amazon.rerank-v1:0` (AWS-native). Cohere also publishes a
rerank model on Bedrock if you need multilingual.

In [ ]:
# Standalone rerank: pull a wider top-K with vector search, then re-score with the rerank API.
# This makes the lift from reranking visible — same query, two orderings side-by-side.

query = "How do I migrate applications to Kubernetes?"

# Step 1 — over-fetch with vector search (top-10 instead of top-3)
baseline = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": query},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 10}},
)
candidates = baseline["retrievalResults"]
print(f"Vector top-10 (cosine scores):")
for i, r in enumerate(candidates, 1):
    print(f"  {i:2d}. score={r.get('score', 0):.4f}  {r['content']['text'][:90]}...")

# Step 2 — rerank with amazon.rerank-v1:0
rerank_model_arn = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.rerank-v1:0"
rerank_resp = bedrock_agent_runtime.rerank(
    queries=[{"type": "TEXT", "textQuery": {"text": query}}],
    sources=[
        {
            "type": "INLINE",
            "inlineDocumentSource": {
                "type": "TEXT",
                "textDocument": {"text": r["content"]["text"]},
            },
        }
        for r in candidates
    ],
    rerankingConfiguration={
        "type": "BEDROCK_RERANKING_MODEL",
        "bedrockRerankingConfiguration": {
            "numberOfResults": 3,
            "modelConfiguration": {"modelArn": rerank_model_arn},
        },
    },
)

print(f"\nReranked top-3 (rerank relevance scores):")
for rank, result in enumerate(rerank_resp["results"], 1):
    orig_idx = result["index"]
    new_score = result["relevanceScore"]
    original_pos = orig_idx + 1
    text = candidates[orig_idx]["content"]["text"][:90]
    moved = "↑" if rank < original_pos else ("↓" if rank > original_pos else "=")
    print(f"  {rank}. score={new_score:.4f}  (was #{original_pos} {moved})  {text}...")

print("\n✓ Standalone rerank working")

### Integrated path: rerank inside `retrieve()`

For production agents you almost never call `rerank()` directly — you let
the KB do over-fetch + rerank + truncate in one round trip. Same API
shape, just attached to `retrievalConfiguration`.

In [ ]:
# KB-integrated rerank: one call, ranked results out.
# numberOfRerankedResults is what you keep AFTER reranking; numberOfResults
# is how wide the initial vector pull goes before reranking.

response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": query},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 10,
            "rerankingConfiguration": {
                "type": "BEDROCK_RERANKING_MODEL",
                "bedrockRerankingConfiguration": {
                    "numberOfRerankedResults": 3,
                    "modelConfiguration": {"modelArn": rerank_model_arn},
                },
            },
        },
    },
)

print(f"Query: '{query}' (with integrated rerank)")
print(f"\nReranked top-{len(response['retrievalResults'])}:")
for i, r in enumerate(response["retrievalResults"], 1):
    print(f"  {i}. score={r.get('score', 0):.4f}  {r['content']['text'][:90]}...")

print("\n✓ Integrated rerank working")
print("\nLatency cost: ~150–300ms added per query.")
print("Dollar cost:   amazon.rerank-v1:0 is ~$1 per 1k queries at typical chunk counts.")

## Step 14: Test Retrieve and Generate (RAG)

In [ ]:
# Full RAG: retrieve + generate answer
rag_query = "What are the best practices for zero trust authentication?"
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": rag_query},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/{config['MODEL_ID']}"
        }
    }
)

answer = response["output"]["text"]
citations = response.get("citations", [])

print(f"Query: '{rag_query}'")
print(f"\nAnswer:\n{answer}")
print(f"\nCitations: {len(citations)} sources referenced")
print("\n\u2713 Retrieve-and-generate working")

## Step 15: Save Configuration

In [ ]:
# KB_ID already loaded from CDK outputs via notebook 01
# config["KB_ID"] is already set
config["KB_DATA_SOURCE_ID"] = ds_id
config["COLLECTION_ID"] = collection_id
config["COLLECTION_ARN"] = collection_arn
config["S3_BUCKET"] = BUCKET_NAME

Path("config.json").write_text(json.dumps(config, indent=2))
print(f"\u2713 Config saved")
print(f"  KB_ID: {kb_id}")
print(f"  KB_DATA_SOURCE_ID: {ds_id}")
print(f"  S3_BUCKET: {BUCKET_NAME}")

## Summary

- [x] S3 bucket created with sample documents
- [x] Aurora pgvector pgvector table (VECTORSEARCH type)
- [x] Security policies (encryption, network, access)
- [x] IAM role for Bedrock Knowledge Base
- [x] Bedrock Knowledge Base with Titan V2 embeddings (1024d)
- [x] S3 data source with chunking configuration
- [x] Ingestion completed successfully
- [x] RAG retrieval and retrieve-and-generate validated

**Next:** [05-guardrails.ipynb](05-guardrails.ipynb) — Create Bedrock Guardrails for content safety